# **Exploratory Data Analysis Notebook**

This notebook explores the cleaned *Sustainability, Wellbeing and Resource Use Dataset*.

The project has already been separated into three stages:

1. **Raw data setup**  
   The `data_raw_setup.ipynb` notebook constructs the merged country-year dataset from the original public sources. It handles source loading, wide-to-long reshaping, year-window selection, country-code harmonisation, and merging.

2. **Data cleaning**  
   The `data_cleaning.ipynb` notebook documents the main cleaning decisions. It deals with duplicate variables, missingness, country exclusions, interpolation, imputation, type corrections, and the final clean dataset export.

3. **Exploratory analysis**  
   This notebook starts from the cleaned dataset and focuses on analytical interpretation. It does not repeat the full setup or cleaning process, except for brief final checks needed to confirm that the dataset is ready for EDA.

## Purpose of this notebook

The purpose of this EDA notebook is to examine whether countries appear to achieve high levels of subjective wellbeing with comparatively lower environmental and material pressure.

The analysis focuses on the relationship between:

- subjective wellbeing;
- consumption-based and production-based CO₂ emissions;
- material footprint;
- energy use;
- income inequality;
- country-level development and regional groupings.

Rather than treating sustainability only as low emissions or low resource use, the notebook focuses on **wellbeing efficiency**: how much reported wellbeing is achieved relative to the environmental and material resources associated with that wellbeing.

## Analytical focus

Although the feature engineering module creates several related variables, this notebook prioritises a smaller set of features that are most central to the project’s research question.

The main EDA features are:

| Feature | Role in the analysis |
|---|---|
| `co2_consumption_production_gap` | Captures the difference between consumption-based and production-based CO₂ emissions per capita. This helps identify countries whose apparent emissions profile changes when accounting for trade and consumption. |
| `happiness_per_consumption_co2` | Measures reported wellbeing per unit of consumption-based CO₂ emissions. This is the central carbon-based wellbeing efficiency indicator. |
| `happiness_per_material_footprint` | Measures reported wellbeing per unit of material footprint. This keeps the material-resource and circular economy dimension central to the analysis. |
| `inequality_adjusted_happiness` | Adjusts reported happiness by income inequality, allowing a more socially cautious interpretation of wellbeing. |
| Year-relative happiness/resource percentiles | Compare countries within the same year, reducing distortion from changes in global distributions over time. |

These features are used to identify countries that combine relatively high wellbeing with lower consumption emissions or material footprint, and to distinguish them from countries whose wellbeing is associated with high resource intensity.

## Notebook structure

The notebook proceeds as follows:

1. **Imports and project setup**  
   Load the required libraries, project paths, and reusable functions.

2. **Load cleaned data**  
   Import the cleaned dataset exported by the cleaning pipeline.

3. **Final validation checks**  
   Confirm dataset shape, missing values, duplicate country-year rows, year coverage, and key variable availability.

4. **Feature preparation for EDA**  
   Build or confirm the engineered features used in the exploratory analysis, including the core wellbeing-efficiency indicators and year-relative percentile variables.

5. **Descriptive overview**  
   Summarise the dataset by year, country coverage, development groups, income groups, and regions.

6. **Core exploratory analysis**  
   Examine production-vs-consumption emissions gaps, wellbeing per consumption CO₂, wellbeing per material footprint, and inequality-adjusted wellbeing.

7. **Country comparison and outlier analysis**  
   Identify countries that perform relatively well or poorly on the central EDA features.

8. **Main findings**  
   Summarise the key empirical patterns and link them back to the project’s research question.

## 1. Imports and project setup

We first import the libraries needed for data manipulation, numerical analysis, and visualisation.

We also set up the project path so that functions from the `src/` modules can be imported directly into the notebook. This keeps the EDA notebook focused on analysis, while reusable logic such as loading data, building features, and plotting remains in the project modules.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Make imports from the scr/ directory work.
import sys
from pathlib import Path

# Add project root to path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

In [3]:
from src.config import CLEAN_PATH
from src.io import load_csv
from src.features import build_features

## 2. Load the cleaned dataset

We now load the cleaned dataset produced by the cleaning pipeline.

At this stage, the dataset has already gone through the main cleaning decisions documented in `data_cleaning.ipynb`. We therefore treat this file as the analytical starting point for EDA, rather than re-cleaning the raw data.

We load the dataset, inspect the first rows, and check its basic shape to confirm that the expected country-year panel has been imported correctly.

In [4]:
# swr_df for "sustainability wellbeing resource"

swr_df = load_csv(CLEAN_PATH)
swr_df.head()

,numeric_code,iso_code,country,year,human_development_groups,hdi_rank_2021,undp_developing_regions,least_devpd_country,landlock_deving_country,small_island_deving_country,...,energy_per_capita,renewables_consumption,temperature_change_from_co2,share_global_co2,land_use_change_co2_per_capita,continent,sub_continent_un,hemisphere,latitude,longitude
0,784,ARE,United Arab Emirates,2013,Very High,26,AS,False,False,False,...,144520.031,0.222,0.002,0.607,-0.014,Asia,Western Asia,Northern Hemisphere,24.0,54.0
1,784,ARE,United Arab Emirates,2014,Very High,26,AS,False,False,False,...,137818.312,0.792,0.002,0.605,-0.011,Asia,Western Asia,Northern Hemisphere,24.0,54.0
2,784,ARE,United Arab Emirates,2015,Very High,26,AS,False,False,False,...,140994.875,0.761,0.002,0.636,-0.009,Asia,Western Asia,Northern Hemisphere,24.0,54.0
3,784,ARE,United Arab Emirates,2016,Very High,26,AS,False,False,False,...,140575.797,0.794,0.002,0.644,-0.007,Asia,Western Asia,Northern Hemisphere,24.0,54.0
4,784,ARE,United Arab Emirates,2017,Very High,26,AS,False,False,False,...,131237.281,1.863,0.002,0.546,-0.007,Asia,Western Asia,Northern Hemisphere,24.0,54.0


In [5]:
swr_df.shape

(558, 29)

In [6]:
swr_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 558 entries, 0 to 557
Data columns (total 29 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   numeric_code                    558 non-null    int64  
 1   iso_code                        558 non-null    str    
 2   country                         558 non-null    str    
 3   year                            558 non-null    int64  
 4   human_development_groups        558 non-null    str    
 5   hdi_rank_2021                   558 non-null    int64  
 6   undp_developing_regions         558 non-null    str    
 7   least_devpd_country             558 non-null    bool   
 8   landlock_deving_country         558 non-null    bool   
 9   small_island_deving_country     558 non-null    bool   
 10  population                      558 non-null    int64  
 11  gdp                             558 non-null    float64
 12  happiness_index                 558 non-null   

The dataset is structured as a country-year panel. Each row corresponds to one country in one year, and the columns contain country identifiers, group classifications, wellbeing indicators, inequality indicators, resource-use variables, and emissions variables.

## 3. Final checks before EDA

Before moving into visual exploration, we run a small number of final validation checks.

The purpose here is not to repeat the full cleaning process. Instead, we check that the cleaned dataset behaves as expected and that the variables needed for the EDA are available and usable.

We focus on five checks:

1. dataset dimensions;
2. duplicate country-year observations;
3. year and country coverage;
4. remaining missing values;
5. availability of the key variables used in the EDA.

These checks help confirm that the dataset is ready for analysis and that any remaining missingness is understood before we start interpreting patterns.

### 3.1 Dataset dimensions

We first check the number of rows and columns in the cleaned dataset.

This gives us a quick confirmation that the imported file has the expected size after the cleaning decisions made in the previous notebook.

In [7]:
n_rows, n_cols = swr_df.shape
print(f"Rows: {n_rows}")
print(f"Columns: {n_cols}")

Rows: 558
Columns: 29


### 3.2 Duplicate country-year rows

Because the dataset is intended to be a country-year panel, each combination of `iso_code` and `year` should appear only once.

We check for duplicated country-year keys. If duplicates exist, this would indicate a structural issue in the merge or cleaning process and should be resolved before EDA.

In [8]:
duplicate_country_years = swr_df.duplicated(subset=["iso_code", "year"]).sum()
print(f"Number of duplicate country-years: {duplicate_country_years}")

Number of duplicate country-years: 0


### 3.3 Year and country coverage

We check the temporal and country coverage of the cleaned dataset.

This helps confirm the analytical window of the dataset and the number of countries included after the missingness-based exclusions made during cleaning.

In [9]:
swr_df["year"].min(), swr_df["year"].max()

(np.int64(2013), np.int64(2021))

In [10]:
swr_df["year"].value_counts().sort_index()

year
2013    62
2014    62
2015    62
2016    62
2017    62
2018    62
2019    62
2020    62
2021    62
Name: count, dtype: int64

In [11]:
swr_df["country"].nunique()

62

In [12]:
swr_df.groupby("year")["country"].nunique()

year
2013    62
2014    62
2015    62
2016    62
2017    62
2018    62
2019    62
2020    62
2021    62
Name: country, dtype: int64

The year-by-year country count allows us to check whether the panel is balanced after cleaning. If the number of countries is constant across years, the dataset can be treated as a balanced country-year panel for the purposes of this EDA.

### 3.4 Remaining missing values

We then check remaining missing values in the cleaned dataset.

Some missing values may be expected and already explained during cleaning. For example, country classification variables may contain values that were recoded as not applicable. However, the main analytical variables used in the EDA should have sufficient coverage.

We therefore inspect both total missing values and missing values by column.

In [13]:
missing_summary = (
    swr_df.isna()
    .sum()
    .to_frame("missing_count")
)

missing_summary["missing_percent"] = (
    missing_summary["missing_count"] / len(swr_df) * 100
)

missing_summary.sort_values("missing_count", ascending=False)

,missing_count,missing_percent
numeric_code,0,0.0
iso_code,0,0.0
country,0,0.0
year,0,0.0
human_development_groups,0,0.0
hdi_rank_2021,0,0.0
undp_developing_regions,0,0.0
least_devpd_country,0,0.0
landlock_deving_country,0,0.0
small_island_deving_country,0,0.0


### 3.5 Key variable availability

We now check the specific variables needed for the EDA.

This is more focused than the general missing-value check. Here, we verify that the wellbeing, emissions, material footprint, energy, and inequality variables needed to construct the main analytical features are present in the dataset.

If any of these variables are missing or contain unexpected missing values, the feature engineering step may fail or produce incomplete indicators.

In [14]:
eda_required_columns = [
    "happiness_index",
    "gini_index",
    "co2_per_capita",
    "consumption_co2_per_capita",
    "material_footprint_per_capita",
    "energy_per_capita",
]

missing_required_columns = [
    col for col in eda_required_columns if col not in swr_df.columns
]

missing_required_columns

[]

In [15]:
swr_df[eda_required_columns].isna().sum()

happiness_index                  0
gini_index                       0
co2_per_capita                   0
consumption_co2_per_capita       0
material_footprint_per_capita    0
energy_per_capita                0
dtype: int64

This check links the cleaned dataset to the analytical feature construction step. These variables are the inputs required for the core EDA features, so confirming their availability reduces the risk of generating misleading feature values.

## 4. Build analytical features

We now build the engineered features used in the EDA, as well as adjacent features that may be useful for future reference.

The aim of this section is to make the analytical logic explicit before relying on the modular feature-building function. We therefore first introduce the main types of features created for the project, then construct the EDA-priority features directly in the notebook so that their meaning is clear.

After this, we run the full `build_features()` function from `src/features.py`. This ensures that all engineered features defined in the project pipeline are available, while keeping the main EDA focused on the smaller subset of features most relevant to the analysis.

### 4.1 Feature types

The feature engineering step creates four broad types of variables.

First, we create **emissions trade/accounting features**. These compare production-based and consumption-based CO₂ emissions. This is useful because countries may appear more or less carbon-intensive depending on whether emissions are counted where goods are produced or where they are consumed.

Second, we create **wellbeing efficiency features**. These measure how much reported wellbeing is achieved per unit of environmental or resource pressure. These are central to the project because the analysis is not only concerned with whether countries have high wellbeing, but whether they achieve it with lower emissions, material footprint, or energy use.

Third, we create **inequality-adjusted wellbeing features**. These adjust reported happiness by income inequality. This allows the analysis to treat wellbeing more cautiously in countries where average happiness may coexist with higher inequality.

Fourth, we create **energy intensity features**. These describe environmental or material pressure relative to energy use. They are mainly diagnostic features and are not the main focus of the EDA, but they may help interpret why countries differ in their wellbeing efficiency.

### 4.2 EDA-priority features

Although several related features are created, the EDA prioritises a smaller subset.

These are the features most directly connected to the project’s main analytical question: whether countries can achieve relatively high wellbeing with lower environmental and material pressure.

The prioritised features are:

| Feature | Why it is prioritised |
|---|---|
| `co2_consumption_production_gap` | Captures the difference between consumption-based and production-based CO₂ emissions per capita. This helps identify whether a country’s apparent emissions profile changes once trade and consumption are considered. |
| `happiness_per_consumption_co2` | Measures reported wellbeing per unit of consumption-based CO₂ emissions. This is the main carbon-based wellbeing efficiency feature. |
| `happiness_per_material_footprint` | Measures reported wellbeing per unit of material footprint. This keeps the material-resource dimension central to the analysis. |
| `inequality_adjusted_happiness` | Adjusts happiness by income inequality, allowing a more cautious interpretation of average wellbeing. |
| Year-relative happiness/resource percentiles | Compare countries within the same year, helping identify cases of high wellbeing with lower resource pressure relative to other countries in that year. |

The remaining engineered features are still useful, but they are treated as supporting or adjacent indicators rather than the main focus of the notebook.

### 4.3 Build the EDA-priority features directly

We first build the prioritised EDA features directly in the notebook.

This makes the analytical logic visible before using the modular `build_features()` function. The aim is not only to create new columns, but to show how each feature follows from the definitions and scales of the variables already available in the cleaned dataset.

We round the engineered features to three decimal places. This does not change the conceptual meaning of the variables, but it keeps the notebook outputs readable and avoids showing unnecessary floating-point precision.

We create a copy of the cleaned dataset before adding features.

This keeps the original cleaned dataframe unchanged and makes `eda_df` the working dataframe for exploratory transformations.

In [16]:
eda_df = swr_df.copy()

#### Consumption-production CO₂ gap

We first calculate the difference between consumption-based and production-based CO₂ emissions per capita.

`co2_per_capita` measures territorial or production-based CO₂ emissions per person. It assigns emissions to the country where they are produced.

`consumption_co2_per_capita` adjusts this by trade. It assigns emissions to the country where goods and services are consumed, even if some of those emissions occurred elsewhere during production.

Because both variables are measured as CO₂ emissions per capita, they are on the same scale and can be subtracted directly.

The feature is calculated as:

`consumption_co2_per_capita - co2_per_capita`

A positive value means that consumption-based emissions are higher than production-based emissions. This suggests that the country’s consumption footprint is larger than its territorial emissions alone would imply.

A negative value means that production-based emissions are higher than consumption-based emissions. This suggests that the country may produce emissions partly associated with goods or services consumed elsewhere.

This feature is useful because it shows how the interpretation of national emissions changes when the accounting basis shifts from production to consumption.

In [17]:
eda_df["co2_consumption_production_gap"] = (
    eda_df["consumption_co2_per_capita"] - eda_df["co2_per_capita"]
).round(3)

We inspect the two emissions variables alongside the new gap feature.

This confirms that the feature has the expected sign and magnitude. Positive values mean consumption-based CO₂ per capita is higher than production-based CO₂ per capita, while negative values mean the reverse.

In [18]:
eda_df[
    [
        "country",
        "year",
        "co2_per_capita",
        "consumption_co2_per_capita",
        "co2_consumption_production_gap",
    ]
].head()

,country,year,co2_per_capita,consumption_co2_per_capita,co2_consumption_production_gap
0,United Arab Emirates,2013,27.348,33.019,5.671
1,United Arab Emirates,2014,26.045,31.416,5.371
2,United Arab Emirates,2015,25.943,30.139,4.196
3,United Arab Emirates,2016,25.235,28.680,3.445
4,United Arab Emirates,2017,21.279,25.843,4.564


#### Happiness per consumption-based CO₂

We calculate happiness per unit of consumption-based CO₂ emissions.

`happiness_index` is the wellbeing numerator. Higher values indicate higher reported wellbeing.

`consumption_co2_per_capita` is the environmental-pressure denominator. It is used here because it reflects emissions associated with consumption, rather than only emissions produced within national borders.

The feature is calculated as:

`happiness_index / consumption_co2_per_capita`

Higher values indicate more reported wellbeing per unit of consumption-based carbon pressure.

We use safe division logic. If the denominator is missing or zero, the result is set to missing. This avoids invalid division and prevents artificial infinite values.

We round the result to three decimal places because this is an exploratory ratio. More decimal precision would make the tables harder to read without adding meaningful interpretive value.

In [19]:
eda_df["happiness_per_consumption_co2"] = np.where(
    (eda_df["consumption_co2_per_capita"].notna())
    & (eda_df["consumption_co2_per_capita"] != 0),
    eda_df["happiness_index"] / eda_df["consumption_co2_per_capita"],
    np.nan,
).round(3)

We inspect happiness, consumption-based CO₂, and the derived ratio together.

This is important because high values of `happiness_per_consumption_co2` can result from high happiness, low consumption-based CO₂, or both.

In [20]:
eda_df[
    [
        "country",
        "year",
        "happiness_index",
        "consumption_co2_per_capita",
        "happiness_per_consumption_co2",
    ]
].head()

,country,year,happiness_index,consumption_co2_per_capita,happiness_per_consumption_co2
0,United Arab Emirates,2013,7.1440,33.019,0.216
1,United Arab Emirates,2014,7.0225,31.416,0.224
2,United Arab Emirates,2015,6.9010,30.139,0.229
3,United Arab Emirates,2016,6.5730,28.680,0.229
4,United Arab Emirates,2017,6.6480,25.843,0.257


#### Happiness per material footprint

We calculate happiness per unit of material footprint.

`material_footprint_per_capita` measures material resource use associated with consumption, expressed per person. This keeps the resource-use and circular-economy dimension central to the EDA.

The feature is calculated as:

`happiness_index / material_footprint_per_capita`

Higher values indicate more reported wellbeing per unit of material resource pressure.

The construction mirrors the carbon-efficiency feature: happiness is placed in the numerator, and a per-capita environmental/resource pressure variable is placed in the denominator.

We again use safe division logic and round the resulting ratio to three decimal places.

In [21]:
eda_df["happiness_per_material_footprint"] = np.where(
    (eda_df["material_footprint_per_capita"].notna())
    & (eda_df["material_footprint_per_capita"] != 0),
    eda_df["happiness_index"] / eda_df["material_footprint_per_capita"],
    np.nan,
).round(3)

We inspect happiness, material footprint, and the derived material-efficiency ratio together.

This helps confirm that the feature is behaving as intended and reminds us that high values may be driven either by high wellbeing or by a low material footprint denominator.

In [22]:
eda_df[
    [
        "country",
        "year",
        "happiness_index",
        "material_footprint_per_capita",
        "happiness_per_material_footprint",
    ]
].head()

,country,year,happiness_index,material_footprint_per_capita,happiness_per_material_footprint
0,United Arab Emirates,2013,7.1440,49.68,0.144
1,United Arab Emirates,2014,7.0225,55.49,0.127
2,United Arab Emirates,2015,6.9010,59.76,0.115
3,United Arab Emirates,2016,6.5730,64.95,0.101
4,United Arab Emirates,2017,6.6480,75.61,0.088


#### Inequality-adjusted happiness

We create an exploratory inequality-adjusted happiness feature.

`happiness_index` measures average subjective wellbeing. However, an average can hide distributional differences.

`gini_index` measures income inequality. In this dataset, it is treated as a 0–100 scale where higher values indicate greater inequality.

We therefore convert the GINI index into a proportional penalty using:

`1 - gini_index / 100`

The full feature is:

`happiness_index * (1 - gini_index / 100)`

For example, if the GINI index is 30, the multiplier is 0.70. If the GINI index is 45, the multiplier is 0.55.

This is not an official wellbeing measure. It is a project-specific exploratory indicator used to make the interpretation of average happiness more cautious where income inequality is higher.

We round the result to three decimal places for readability.

In [23]:
eda_df["inequality_adjusted_happiness"] = (
    eda_df["happiness_index"] * (1 - eda_df["gini_index"] / 100)
).round(3)

We inspect raw happiness, the GINI index, and the inequality-adjusted happiness feature together.

This confirms that higher inequality reduces the adjusted happiness score relative to the original happiness index.

In [24]:
eda_df[
    [
        "country",
        "year",
        "happiness_index",
        "gini_index",
        "inequality_adjusted_happiness",
    ]
].head()

,country,year,happiness_index,gini_index,inequality_adjusted_happiness
0,United Arab Emirates,2013,7.1440,32.50,4.822
1,United Arab Emirates,2014,7.0225,31.28,4.826
2,United Arab Emirates,2015,6.9010,30.06,4.827
3,United Arab Emirates,2016,6.5730,28.84,4.677
4,United Arab Emirates,2017,6.6480,27.62,4.812


#### CO₂ per 1,000 units of energy

We also create one diagnostic energy-intensity feature.

The main wellbeing-efficiency features compare happiness with environmental or material pressure. However, it is also useful to inspect how carbon-intensive energy use appears to be. This can help interpret why some countries may perform better or worse on the main wellbeing-efficiency indicators.

For this purpose, we create a feature based on:

`co2_per_capita / energy_per_capita`

`co2_per_capita` measures production-based CO₂ emissions per person. `energy_per_capita` measures energy use per person. Dividing one by the other gives an indication of CO₂ emissions relative to energy use.

However, `energy_per_capita` is measured on a much larger numerical scale than `co2_per_capita`. If we use the raw ratio, the resulting values are very small and are difficult to read in notebook outputs, especially after rounding.

We therefore rescale the feature by multiplying the ratio by 1,000:

`(co2_per_capita / energy_per_capita) * 1000`

This does not change the ordering of countries or the substantive meaning of the feature. It simply expresses the result per 1,000 units of energy use, which makes the values easier to inspect.

This feature is diagnostic rather than central to the EDA. It does not measure wellbeing efficiency directly, but it may help explain whether high or low emissions are partly related to the carbon intensity of energy use.

In [25]:
eda_df["co2_per_1000_energy"] = np.where(
    (eda_df["energy_per_capita"].notna())
    & (eda_df["energy_per_capita"] != 0),
    (eda_df["co2_per_capita"] / eda_df["energy_per_capita"]) * 1000,
    np.nan,
).round(3)

We inspect the input variables and the rescaled feature together.

This confirms that the feature gives readable values while preserving its interpretation as CO₂ emissions relative to energy use.

In [26]:
eda_df[
    [
        "country",
        "year",
        "co2_per_capita",
        "energy_per_capita",
        "co2_per_1000_energy",
    ]
].head()

,country,year,co2_per_capita,energy_per_capita,co2_per_1000_energy
0,United Arab Emirates,2013,27.348,144520.031,0.189
1,United Arab Emirates,2014,26.045,137818.312,0.189
2,United Arab Emirates,2015,25.943,140994.875,0.184
3,United Arab Emirates,2016,25.235,140575.797,0.180
4,United Arab Emirates,2017,21.279,131237.281,0.162


We also inspect the summary statistics.

This helps check whether the rescaled feature produces a useful range of values after rounding to three decimal places.

In [27]:
eda_df["co2_per_1000_energy"].describe()

count    558.000000
mean       0.199746
std        0.052522
min        0.061000
25%        0.172000
50%        0.197500
75%        0.224750
max        0.477000
Name: co2_per_1000_energy, dtype: float64

### 4.4 Collect and inspect the EDA-priority features

We now collect the EDA-priority features in a single list.

This makes it easier to inspect the variables that will be used most directly in the exploratory analysis. These are not the only engineered features in the project, but they are the ones most closely connected to the notebook’s main analytical focus.

We first summarise the prioritised features.

This helps us check whether the constructed variables have plausible ranges and whether any values need closer inspection before they are used in visualisations. This is especially important for ratio features, because high values can sometimes be driven by very low denominator values.

In [28]:
eda_priority_features = [
    "co2_consumption_production_gap",
    "happiness_per_consumption_co2",
    "happiness_per_material_footprint",
    "inequality_adjusted_happiness",
]

In [29]:
eda_df[eda_priority_features].describe()

,co2_consumption_production_gap,happiness_per_consumption_co2,happiness_per_material_footprint,inequality_adjusted_happiness
count,558.000000,558.000000,558.000000,558.000000
mean,0.831873,1.256586,0.419296,3.971989
std,2.737232,1.221287,0.309448,0.808297
min,-17.668000,0.216000,0.088000,1.801000
25%,0.023000,0.666750,0.232000,3.291250
50%,0.459000,0.805000,0.327000,3.912500
75%,1.639750,1.378000,0.534750,4.631500
max,15.711000,11.717000,2.209000,5.693000


We also inspect the prioritised features alongside the country and year identifiers.

This gives a more concrete view of the constructed values and confirms that the new features have been added correctly to the EDA dataframe.

In [30]:
eda_df[
    ["country", "year"] + eda_priority_features
].head()

,country,year,co2_consumption_production_gap,happiness_per_consumption_co2,happiness_per_material_footprint,inequality_adjusted_happiness
0,United Arab Emirates,2013,5.671,0.216,0.144,4.822
1,United Arab Emirates,2014,5.371,0.224,0.127,4.826
2,United Arab Emirates,2015,4.196,0.229,0.115,4.827
3,United Arab Emirates,2016,3.445,0.229,0.101,4.677
4,United Arab Emirates,2017,4.564,0.257,0.088,4.812


Finally, we inspect the highest values of `happiness_per_consumption_co2`.

This is a useful sanity check because ratio-based features can be sensitive to low denominator values. Before interpreting high wellbeing-efficiency scores substantively, we need to check whether they reflect meaningful combinations of high happiness and lower emissions, or whether they are mainly driven by very low consumption-based CO₂ values.

In [31]:
eda_df[
    [
        "country",
        "year",
        "happiness_index",
        "consumption_co2_per_capita",
        "happiness_per_consumption_co2",
    ]
].sort_values("happiness_per_consumption_co2", ascending=False).head(10)

,country,year,happiness_index,consumption_co2_per_capita,happiness_per_consumption_co2
45,Bangladesh,2013,4.804,0.410,11.717
47,Bangladesh,2015,4.694,0.482,9.739
46,Bangladesh,2014,4.749,0.676,7.025
405,Pakistan,2013,5.292,0.783,6.759
407,Pakistan,2015,5.194,0.790,6.575
406,Pakistan,2014,5.243,0.798,6.570
411,Pakistan,2019,5.653,0.950,5.951
48,Bangladesh,2016,4.643,0.787,5.900
49,Bangladesh,2017,4.608,0.826,5.579
412,Pakistan,2020,5.693,1.024,5.560


We repeat the same check for `happiness_per_material_footprint`.

This helps us identify whether high material-efficiency scores are associated with relatively high happiness, relatively low material footprint, or both.

In [32]:
eda_df[
    [
        "country",
        "year",
        "happiness_index",
        "material_footprint_per_capita",
        "happiness_per_material_footprint",
    ]
].sort_values("happiness_per_material_footprint", ascending=False).head(10)

,country,year,happiness_index,material_footprint_per_capita,happiness_per_material_footprint
46,Bangladesh,2014,4.749,2.15,2.209
48,Bangladesh,2016,4.643,2.33,1.993
49,Bangladesh,2017,4.608,2.37,1.944
47,Bangladesh,2015,4.694,2.54,1.848
405,Pakistan,2013,5.292,2.90,1.825
53,Bangladesh,2021,5.025,2.88,1.745
406,Pakistan,2014,5.243,3.06,1.713
407,Pakistan,2015,5.194,3.09,1.681
52,Bangladesh,2020,4.833,2.88,1.678
45,Bangladesh,2013,4.804,2.91,1.651


### 4.5 Run the full feature-building function

The features above were built directly in the notebook so that their functional form and interpretation are visible.

We now run the full `build_features()` function from `src/features.py`. This applies the modular version of the feature engineering step and creates the complete set of engineered features defined for the project.

This is important because the notebook should explain the central analytical features, while the module should remain the reproducible source for the full feature-building pipeline.

In [33]:
eda_df = build_features(swr_df)

In [34]:
feature_columns = [
    "co2_consumption_production_gap",
    "co2_consumption_production_ratio",
    "happiness_per_co2",
    "happiness_per_consumption_co2",
    "happiness_per_material_footprint",
    "happiness_per_1000_energy",
    "inequality_adjusted_happiness",
    "ineq_adj_happiness_per_co2",
    "ineq_adj_happiness_per_consumption_co2",
    "ineq_adj_happiness_per_material_footprint",
    "ineq_adj_happiness_per_1000_energy",
    "co2_per_1000_energy",
    "material_footprint_per_1000_energy",
]

missing_feature_columns = [
    col for col in feature_columns if col not in eda_df.columns
]

missing_feature_columns

[]

In [35]:
eda_df[feature_columns].head()

,co2_consumption_production_gap,co2_consumption_production_ratio,happiness_per_co2,happiness_per_consumption_co2,happiness_per_material_footprint,happiness_per_1000_energy,inequality_adjusted_happiness,ineq_adj_happiness_per_co2,ineq_adj_happiness_per_consumption_co2,ineq_adj_happiness_per_material_footprint,ineq_adj_happiness_per_1000_energy,co2_per_1000_energy,material_footprint_per_1000_energy
0,5.671,1.207,0.261,0.216,0.144,0.049,4.822,0.176,0.146,0.097,0.033,0.189,0.344
1,5.371,1.206,0.270,0.224,0.127,0.051,4.826,0.185,0.154,0.087,0.035,0.189,0.403
2,4.196,1.162,0.266,0.229,0.115,0.049,4.827,0.186,0.160,0.081,0.034,0.184,0.424
3,3.445,1.137,0.260,0.229,0.101,0.047,4.677,0.185,0.163,0.072,0.033,0.180,0.462
4,4.564,1.214,0.312,0.257,0.088,0.051,4.812,0.226,0.186,0.064,0.037,0.162,0.576


We also check whether the modular feature-building step introduced missing values in the engineered features.

Missing values can occur where the original inputs are missing or where a denominator is zero. This is expected behaviour for safely constructed ratio features, but it should still be checked before analysis.

In [36]:
eda_df[feature_columns].isna().sum()

co2_consumption_production_gap               0
co2_consumption_production_ratio             0
happiness_per_co2                            0
happiness_per_consumption_co2                0
happiness_per_material_footprint             0
happiness_per_1000_energy                    0
inequality_adjusted_happiness                0
ineq_adj_happiness_per_co2                   0
ineq_adj_happiness_per_consumption_co2       0
ineq_adj_happiness_per_material_footprint    0
ineq_adj_happiness_per_1000_energy           0
co2_per_1000_energy                          0
material_footprint_per_1000_energy           0
dtype: int64

Finally, we inspect the summary statistics of the full engineered feature set.

This provides a quick check of the scale and range of the created variables after the modular build has been applied.

In [37]:
eda_df[feature_columns].describe()

,co2_consumption_production_gap,co2_consumption_production_ratio,happiness_per_co2,happiness_per_consumption_co2,happiness_per_material_footprint,happiness_per_1000_energy,inequality_adjusted_happiness,ineq_adj_happiness_per_co2,ineq_adj_happiness_per_consumption_co2,ineq_adj_happiness_per_material_footprint,ineq_adj_happiness_per_1000_energy,co2_per_1000_energy,material_footprint_per_1000_energy
count,558.000000,558.000000,558.000000,558.000000,558.000000,558.000000,558.000000,558.000000,558.000000,558.000000,558.000000,558.000000,558.000000
mean,0.831873,1.209814,1.513955,1.256586,0.419296,0.300833,3.971989,0.963004,0.796351,0.268514,0.190729,0.199746,0.687889
std,2.737232,0.427824,1.549797,1.221287,0.309448,0.329713,0.808297,1.002161,0.789843,0.205214,0.212960,0.052522,0.335245
min,-17.668000,0.607000,0.148000,0.216000,0.088000,0.027000,1.801000,0.096000,0.141000,0.064000,0.017000,0.061000,0.159000
25%,0.023000,1.003000,0.704250,0.666750,0.232000,0.140000,3.291250,0.466500,0.429500,0.150250,0.091000,0.172000,0.466000
50%,0.459000,1.119000,1.020500,0.805000,0.327000,0.192000,3.912500,0.694000,0.551000,0.219500,0.130000,0.197500,0.616500
75%,1.639750,1.307750,1.591500,1.378000,0.534750,0.286750,4.631500,1.007500,0.810500,0.315000,0.172000,0.224750,0.829750
max,15.711000,4.354000,11.604000,11.717000,2.209000,2.453000,5.693000,7.845000,7.922000,1.493000,1.658000,0.477000,1.988000


## 5. Dataset overview

In [38]:
# Numerical variables
df.describe()

# Categorical values (Example for a columnn)
# df['category_column'].value_counts()

NameError: name 'df' is not defined

## 6. Production-based vs consumption-based emissions

## 7. Wellbeing efficiency: consumption CO₂ and material footprint

## 8. Inequality-adjusted wellbeing

## 9. Year-relative percentile comparisons

## 10. Main exploratory findings

## 7. Análisis univariado de variables numéricas
Realizar histogramas y boxplots para analizar la distribución de las variables numéricas.

In [ ]:
# Histogram for a numerical variable
# df['numeric_column'].hist()
# plt.show()

# Boxplot
# df.boxplot(column='numeric_column')
# plt.show()

## 8. Análisis univariado de variables categóricas
Realizar gráficos de barras para analizar la frecuencia de las variables categóricas.

In [ ]:
# Bar plot for a categorical variable
# df['categorical_column'].value_counts().plot(kind='bar')
# plt.show()

## 9. Análisis bivariado entre variables
Explorar relaciones entre variables numéricas y categóricas mediante scatterplots y tablas cruzadas.

In [ ]:
# Scatterplot
# plt.scatter(df['numeric_column_x'], df['numeric_column_y'])
# plt.show()

# Crosstab for two categorical variables   
# pd.crosstab(df['categorical_column1'], df['categorical_column2'])

## 10. Visualización de correlaciones
Calcular y visualizar la matriz de correlación entre variables numéricas usando un heatmap.

In [ ]:
# Correlation matrix
corr = df.corr()
sns.heatmap(corr, annot=True)
plt.show()